# Dysgraphia Detection Project - Initialization

## Project Overview

This notebook contains **all initialization and data preprocessing steps** for the dysgraphia detection project using the Drotar dataset.

### What This Notebook Does:
1. **Configuration Setup**: Defines global settings and hyperparameters
2. **Data Loading**: Functions to load metadata and raw time-series files
3. **Feature Engineering**: Computes derivative features (velocity, acceleration, jerk)
4. **Data Preprocessing**: Normalization, padding, and truncation utilities
5. **Dataset Class**: PyTorch Dataset wrapper for handwriting data
6. **Data Splitting**: Subject-independent train/test splits

### How to Use:
1. **Run this notebook first** before any model training notebooks
2. All functions and classes defined here will be available in other notebooks
3. Make sure to set correct paths in the `Config` class (Cell 1)

### Next Steps:
After running this notebook, proceed to:
- `01_model_cnn1d.ipynb` - Train 1D CNN model
- `02_model_tcn.ipynb` - Train TCN model  
- `03_model_cnnlstm.ipynb` - Train CNN-LSTM model
- `04_explainability.ipynb` - Feature importance and explainability analysis

## 1. Imports and Configuration

This section imports all necessary libraries and defines the global configuration class.

**Important**: Update the `DATA_ROOT` and `META_XLSX` paths to match your local setup!

In [25]:
# --- 1.1 Install Required Packages (if needed) ---

# Install missing packages automatically
import subprocess
import sys

required_packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'torch': 'torch',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'tqdm': 'tqdm',
    'openpyxl': 'openpyxl'  # Required for reading Excel files (.xlsx)
}

missing_packages = []
for module_name, package_name in required_packages.items():
    try:
        __import__(module_name)
    except ImportError:
        missing_packages.append(package_name)

# Also check for openpyxl specifically (needed for pandas Excel reading)
try:
    import openpyxl
except ImportError:
    if 'openpyxl' not in missing_packages:
        missing_packages.append('openpyxl')

if missing_packages:
    print(f"Installing missing packages: {', '.join(missing_packages)}")
    for package in missing_packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])
    print("✓ Packages installed successfully!")
    print("⚠ Please restart the kernel after installation if you encounter any issues.")
else:
    print("✓ All required packages are already installed!")

# --- 1.2 Import Required Libraries ---

import os
import re
import zipfile
import numpy as np
import pandas as pd
from typing import List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

import matplotlib.pyplot as plt
from tqdm import tqdm

print("✓ All libraries imported successfully!")

✓ All required packages are already installed!
✓ All libraries imported successfully!


### Configuration Class

The `Config` class contains all hyperparameters and paths. **Modify these values according to your setup!**

In [26]:
# --- 1.2 Global Configuration ---

class Config:
    """
    Global configuration for the dysgraphia detection project.
    
    IMPORTANT: Update DATA_ROOT and META_XLSX paths to match your local setup!
    """
    # ========== Data Paths ==========
    # Update these paths to match your local file locations
    DATA_ROOT = r"C:\Users\tiama\OneDrive\Desktop\period 2 courses\Reserach Project\DysXAI\dataSciRep_public"
    META_XLSX = r"C:\Users\tiama\OneDrive\Desktop\period 2 courses\Reserach Project\DysXAI\data2_SciRep_pub.xlsx"
    
    # ========== Time-Series Configuration ==========
    MAX_LEN = 2000            # Fixed sequence length (pad/truncate to this length)
    USE_DERIVATIVES = True    # Add velocity, acceleration, and jerk features
    
    # ========== Training Hyperparameters ==========
    BATCH_SIZE = 16
    NUM_EPOCHS = 50
    LR = 1e-3                 # Learning rate
    WEIGHT_DECAY = 1e-4       # L2 regularization
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    # ========== Cross-Validation Settings ==========
    # 3-way split ratios: Train (70%), Validation (15%), Test (15%)
    TRAIN_SUBJECT_RATIO = 0.7   # 70% subjects for training
    VAL_SUBJECT_RATIO = 0.15    # 15% subjects for validation
    TEST_SUBJECT_RATIO = 0.15   # 15% subjects for testing (automatically calculated)
    NUM_REPEATED_SPLITS = 5     # Number of repeated subject-independent splits
    INNER_CV_FOLDS = 3          # Number of folds for inner cross-validation (hyperparameter tuning)
    
    # ========== Early Stopping ==========
    EARLY_STOPPING_PATIENCE = 5   # Stop training if validation loss doesn't improve for N epochs
    EARLY_STOPPING_ENABLED = True
    
    # ========== Hyperparameter Search Space ==========
    LR_SEARCH_SPACE = [1e-3]      # Learning rates to try during hyperparameter search
    DROPOUT_SEARCH_SPACE = [0.3]  # Dropout rates to try
    
    # ========== Miscellaneous ==========
    RANDOM_STATE = 42  # Random seed for reproducibility

print("=" * 60)
print("CONFIGURATION LOADED")
print("=" * 60)
print(f"Device: {Config.DEVICE}")
print(f"Data Root: {Config.DATA_ROOT}")
print(f"Metadata File: {Config.META_XLSX}")
print(f"Max Sequence Length: {Config.MAX_LEN}")
print(f"Use Derivatives: {Config.USE_DERIVATIVES}")
print("=" * 60)

CONFIGURATION LOADED
Device: cpu
Data Root: C:\Users\tiama\OneDrive\Desktop\period 2 courses\Reserach Project\DysXAI\dataSciRep_public
Metadata File: C:\Users\tiama\OneDrive\Desktop\period 2 courses\Reserach Project\DysXAI\data2_SciRep_pub.xlsx
Max Sequence Length: 2000
Use Derivatives: True


## 2. Data Loading Functions

This section contains functions to:
- Load metadata from Excel file
- Load raw time-series data from files
- Discover and map files to subject IDs

**Note**: These functions handle the Drotar dataset format. Adapt if using a different dataset structure.

In [27]:
# --- 2.1 Load Metadata from Excel ---

def load_metadata(meta_path: str) -> pd.DataFrame:
    """
    Load metadata from Excel file and standardize columns.
    
    Args:
        meta_path: Path to the Excel metadata file
        
    Returns:
        DataFrame with columns: ['file_name', 'subject_id', 'label']
        - file_name: Placeholder (actual files discovered later)
        - subject_id: Subject identifier
        - label: 0 (non-dysgraphic) or 1 (dysgraphic)
    """
    df = pd.read_excel(meta_path)
    
    # Map Excel columns to standardized names
    # Based on data2_SciRep_pub.xlsx structure:
    # - 'ID' column contains subject identifiers
    # - 'diag' column contains diagnosis ('DYSGR' for dysgraphic)
    df['subject_id'] = df['ID'].astype(int)
    df['label'] = (df['diag'].astype(str).str.upper() == 'DYSGR').astype(int)
    df['file_name'] = df['subject_id'].astype(str)  # Placeholder
    
    # Return only required columns
    required_cols = ['file_name', 'subject_id', 'label']
    return df[required_cols].copy()

print("✓ Metadata loader function defined")

✓ Metadata loader function defined


In [28]:
# --- 2.2 Load Raw Time-Series Data ---

def load_raw_timeseries(filepath: str) -> np.ndarray:
    """
    Load a single handwriting sample from a file (.svc or .txt format).
    
    Expected columns: x, y, t, pressure, azimuth, altitude, pen_status
    
    Args:
        filepath: Path to the time-series file
        
    Returns:
        np.ndarray of shape (T, 7) where T is the sequence length
    """
    data = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue  # Skip empty lines and comments
            
            parts = line.split()
            try:
                row = [float(v) for v in parts]
                data.append(row)
            except ValueError:
                continue  # Skip non-numeric lines
    
    if len(data) == 0:
        raise ValueError(f"No numeric data found in file: {filepath}")
    
    # Keep only rows with the most frequent length (handles inconsistent files)
    lengths = [len(r) for r in data]
    uniq, counts = np.unique(lengths, return_counts=True)
    target_len = int(uniq[np.argmax(counts)])
    filtered = [r for r in data if len(r) == target_len]
    
    if len(filtered) == 0:
        raise ValueError(f"All rows in {filepath} had inconsistent lengths")
    
    arr = np.array(filtered, dtype=np.float32)
    
    # Standardize to 7 channels (x, y, t, pressure, azimuth, altitude, pen_status)
    EXPECTED_BASE_CHANNELS = 7
    current_channels = arr.shape[1]
    
    if current_channels > EXPECTED_BASE_CHANNELS:
        arr = arr[:, :EXPECTED_BASE_CHANNELS]  # Truncate
    elif current_channels < EXPECTED_BASE_CHANNELS:
        # Pad with zeros
        padding_needed = EXPECTED_BASE_CHANNELS - current_channels
        arr = np.pad(arr, ((0, 0), (0, padding_needed)), 'constant', constant_values=0)
    
    return arr  # shape (T, 7)

print("✓ Raw time-series loader function defined")

✓ Raw time-series loader function defined


In [29]:
# --- 2.3 Discover Files and Map to Subject IDs ---

def discover_files_and_map_subjects(data_root: str, subject_meta: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """
    Discover all .svc/.txt files in data_root and map them to subject IDs.
    
    Args:
        data_root: Root directory containing the data files
        subject_meta: DataFrame with 'subject_id' and 'label' columns
        verbose: Whether to print debugging information
        
    Returns:
        DataFrame with columns: ['subject_id', 'file_name', 'label']
    """
    # Check if directory exists
    if not os.path.exists(data_root):
        raise FileNotFoundError(f"Data directory not found: {data_root}\nPlease check Config.DATA_ROOT path!")
    
    file_records = []
    subject_ids = set(subject_meta['subject_id'].tolist())
    
    if verbose:
        print(f"Looking for files in: {data_root}")
        print(f"Looking for {len(subject_ids)} subject IDs: {sorted(list(subject_ids))[:10]}..." if len(subject_ids) > 10 else f"Looking for {len(subject_ids)} subject IDs: {sorted(list(subject_ids))}")
    
    # Count files found
    total_files_found = 0
    files_with_numbers = 0
    matched_files = 0
    
    # Walk through all directories and find matching files
    for root, dirs, files in os.walk(data_root):
        for fname in files:
            if not fname.lower().endswith(('.svc', '.txt')):
                continue
            
            total_files_found += 1
            base = os.path.splitext(fname)[0]
            
            # Extract integer substrings from filename
            ints_in_name = [int(m.group()) for m in re.finditer(r'\d+', base)]
            if not ints_in_name:
                continue
            
            files_with_numbers += 1
            
            # Match to subject IDs - try different matching strategies
            matched = False
            for sid in ints_in_name:
                if sid in subject_ids:
                    rel_path = os.path.relpath(os.path.join(root, fname), data_root)
                    file_records.append({
                        'subject_id': sid,
                        'file_name': rel_path,
                    })
                    matched_files += 1
                    matched = True
                    break  # Stop at first match
            
            # If no direct match, try matching with leading zeros (e.g., "00008" matches 8)
            if not matched:
                for sid in subject_ids:
                    # Try matching with zero-padded numbers
                    for num in ints_in_name:
                        # Check if filename contains subject ID with various padding
                        sid_str = str(sid)
                        if sid_str in base or f"{sid:05d}" in base or f"{sid:04d}" in base or f"{sid:03d}" in base:
                            rel_path = os.path.relpath(os.path.join(root, fname), data_root)
                            file_records.append({
                                'subject_id': sid,
                                'file_name': rel_path,
                            })
                            matched_files += 1
                            matched = True
                            break
                    if matched:
                        break
    
    if verbose:
        print(f"\nFile Discovery Summary:")
        print(f"  Total .svc/.txt files found: {total_files_found}")
        print(f"  Files with numbers in name: {files_with_numbers}")
        print(f"  Files matched to subject IDs: {matched_files}")
    
    file_df = pd.DataFrame(file_records)
    
    if file_df.empty:
        print(f"\n⚠ ERROR: No files matched!")
        print(f"\nDebugging information:")
        print(f"  - Data directory exists: {os.path.exists(data_root)}")
        print(f"  - Subject IDs in metadata: {sorted(list(subject_ids))[:20]}...")
        
        # Show some example files found
        example_files = []
        for root, dirs, files in os.walk(data_root):
            for fname in files[:5]:  # Show first 5 files
                if fname.lower().endswith(('.svc', '.txt')):
                    example_files.append(os.path.join(root, fname))
            if len(example_files) >= 5:
                break
        
        if example_files:
            print(f"\n  Example files found:")
            for f in example_files[:5]:
                print(f"    - {f}")
        else:
            print(f"\n  ⚠ No .svc or .txt files found in directory!")
            print(f"  Please check:")
            print(f"    1. Is the data unzipped?")
            print(f"    2. Is Config.DATA_ROOT path correct?")
            print(f"    3. Are files in .svc or .txt format?")
        
        raise RuntimeError(f"No .svc/.txt files in {data_root} matched any subject IDs")
    
    # Validate files have enough channels (>= 3 for x, y, t)
    if verbose:
        print(f"\nValidating files (checking channels)...")
    
    valid_records = []
    for idx, row in file_df.iterrows():
        full_path = os.path.join(data_root, row['file_name'])
        try:
            ts = load_raw_timeseries(full_path)
            if ts.shape[1] >= 3:  # Need at least x, y, t
                valid_records.append(row)
        except Exception as e:
            if verbose:
                print(f"  Skipping {row['file_name']}: {e}")
            continue
    
    file_df = pd.DataFrame(valid_records)
    
    if file_df.empty:
        raise RuntimeError("No valid files found after validation")
    
    if verbose:
        print(f"  Valid files: {len(file_df)}")
    
    # Merge with subject labels
    meta_df = file_df.merge(subject_meta, on='subject_id', how='inner')
    
    return meta_df

print("✓ File discovery function defined")

✓ File discovery function defined


## 3. Feature Engineering

This section computes **derivative features** from the raw time-series data:
- **Velocity**: First derivative of x and y positions
- **Acceleration**: Second derivative (rate of change of velocity)
- **Jerk**: Third derivative (rate of change of acceleration)

These features capture important kinematic properties of handwriting that are often indicators of dysgraphia.

In [30]:
# --- 3.1 Compute Derivative Features ---

def compute_derivatives(sample: np.ndarray, x_idx=0, y_idx=1, t_idx=2) -> np.ndarray:
    """
    Compute velocity, acceleration, and jerk for x and y coordinates.
    
    These derivative features capture kinematic properties of handwriting:
    - Velocity: How fast the pen is moving
    - Acceleration: How the speed is changing
    - Jerk: Smoothness of movement (important for dysgraphia detection)
    
    Args:
        sample: Time-series array of shape (T, C) with at least x, y, t columns
        x_idx: Column index for x-coordinate (default: 0)
        y_idx: Column index for y-coordinate (default: 1)
        t_idx: Column index for time (default: 2)
        
    Returns:
        Augmented array of shape (T, C + 6) with [vx, vy, ax, ay, jx, jy] appended
    """
    T, C = sample.shape
    
    if max(x_idx, y_idx, t_idx) >= C:
        raise ValueError(f"Not enough channels ({C}) to compute derivatives; need at least 3.")
    
    # Handle very short sequences
    if T < 2:
        zero_derivs = np.zeros((T, 6), dtype=np.float32)
        return np.concatenate([sample, zero_derivs], axis=-1)
    
    # Extract coordinates
    x = sample[:, x_idx]
    y = sample[:, y_idx]
    t = sample[:, t_idx]
    
    # Compute time differences (avoid division by zero)
    dt = np.diff(t)
    dt[dt == 0] = 1e-6
    
    # Velocity (first derivative)
    vx = np.diff(x) / dt
    vy = np.diff(y) / dt
    # Pad first value to maintain length
    vx = np.concatenate([[vx[0]], vx])
    vy = np.concatenate([[vy[0]], vy])
    
    # Acceleration (second derivative)
    ax = np.diff(vx) / dt
    ay = np.diff(vy) / dt
    ax = np.concatenate([[ax[0]], ax])
    ay = np.concatenate([[ay[0]], ay])
    
    # Jerk (third derivative) - measures smoothness
    jx = np.diff(ax) / dt
    jy = np.diff(ay) / dt
    jx = np.concatenate([[jx[0]], jx])
    jy = np.concatenate([[jy[0]], jy])
    
    # Stack derivatives: [vx, vy, ax, ay, jx, jy]
    derivs = np.stack([vx, vy, ax, ay, jx, jy], axis=-1)  # (T, 6)
    
    # Concatenate with original features
    augmented = np.concatenate([sample, derivs], axis=-1)
    return augmented

print("✓ Derivative computation function defined")

✓ Derivative computation function defined


## 4. Data Preprocessing

This section contains utilities for:
- **Padding/Truncation**: Standardize sequence lengths
- **Normalization**: Scale features using StandardScaler (fit on training data only)

In [31]:
# --- 4.1 Padding and Truncation ---

def pad_truncate(ts: np.ndarray, max_len: int) -> Tuple[np.ndarray, int]:
    """
    Pad or truncate a time series to a fixed length.
    
    This ensures all sequences have the same length for batch processing.
    - Sequences shorter than max_len are padded with zeros
    - Sequences longer than max_len are truncated
    
    Args:
        ts: Time-series array of shape (T, C)
        max_len: Target sequence length
        
    Returns:
        padded: Array of shape (max_len, C)
        length: Original length (clipped to max_len)
    """
    T, C = ts.shape
    length = min(T, max_len)
    padded = np.zeros((max_len, C), dtype=np.float32)
    padded[:length] = ts[:length]
    return padded, length

print("✓ Padding/truncation function defined")

✓ Padding/truncation function defined


In [32]:
# --- 4.2 Mixed Feature Scaling (MinMax for X/Y, StandardScaler for rest) ---

class MixedFeatureScaler:
    """
    Custom scaler that applies different scaling strategies to different features.
    
    - X and Y features (indices 0 and 1): MinMax scaling to [0, 1] range
    - All other features (Time, Pressure, Kinematics): StandardScaler (mean=0, std=1)
    
    IMPORTANT: Fit only on training data to prevent data leakage!
    """
    
    def __init__(self, x_idx: int = 0, y_idx: int = 1):
        """
        Initialize the mixed feature scaler.
        
        Args:
            x_idx: Column index for X coordinate (default: 0)
            y_idx: Column index for Y coordinate (default: 1)
        """
        self.x_idx = x_idx
        self.y_idx = y_idx
        self.x_min = None
        self.x_max = None
        self.y_min = None
        self.y_max = None
        self.standard_scaler = StandardScaler()
        self.is_fitted = False
    
    def fit(self, X_list: List[np.ndarray]):
        """
        Fit the scaler on training data only.
        
        For X and Y: Compute global min/max across all training samples.
        For other features: Fit StandardScaler on all training samples.
        
        Args:
            X_list: List of training time-series arrays, each of shape (T, C)
        """
        # Concatenate all training samples
        concat = np.concatenate(X_list, axis=0)  # (sum_T, C)
        num_features = concat.shape[1]
        
        # Compute min/max for X and Y features
        self.x_min = float(concat[:, self.x_idx].min())
        self.x_max = float(concat[:, self.x_idx].max())
        self.y_min = float(concat[:, self.y_idx].min())
        self.y_max = float(concat[:, self.y_idx].max())
        
        # Handle edge case: if min == max, set a small range to avoid division by zero
        if self.x_max == self.x_min:
            self.x_max = self.x_min + 1e-6
        if self.y_max == self.y_min:
            self.y_max = self.y_min + 1e-6
        
        # Fit StandardScaler on all features (we'll only use it for non-X/Y features)
        # Create a mask for non-X/Y features
        other_indices = [i for i in range(num_features) if i not in [self.x_idx, self.y_idx]]
        
        if len(other_indices) > 0:
            # Fit StandardScaler on non-X/Y features
            other_features = concat[:, other_indices]
            self.standard_scaler.fit(other_features)
        
        self.is_fitted = True
    
    def transform(self, X: np.ndarray) -> np.ndarray:
        """
        Apply the fitted scaler to a single time-series sample.
        
        Args:
            X: Time-series array of shape (T, C)
            
        Returns:
            Scaled array of shape (T, C)
        """
        if not self.is_fitted:
            raise ValueError("Scaler must be fitted before transform!")
        
        T, C = X.shape
        X_scaled = X.copy()
        
        # Apply MinMax scaling to X and Y features
        X_scaled[:, self.x_idx] = (X[:, self.x_idx] - self.x_min) / (self.x_max - self.x_min)
        X_scaled[:, self.y_idx] = (X[:, self.y_idx] - self.y_min) / (self.y_max - self.y_min)
        
        # Apply StandardScaler to all other features
        other_indices = [i for i in range(C) if i not in [self.x_idx, self.y_idx]]
        if len(other_indices) > 0:
            other_features = X[:, other_indices]
            other_scaled = self.standard_scaler.transform(other_features)
            X_scaled[:, other_indices] = other_scaled
        
        return X_scaled


def fit_scaler_on_train(X_list: List[np.ndarray]) -> MixedFeatureScaler:
    """
    Fit a MixedFeatureScaler on training data.
    
    IMPORTANT: Only fit on training data to avoid data leakage!
    
    Args:
        X_list: List of training time-series arrays, each of shape (T, C)
        
    Returns:
        Fitted MixedFeatureScaler object
    """
    scaler = MixedFeatureScaler(x_idx=0, y_idx=1)
    scaler.fit(X_list)
    return scaler

print("✓ MixedFeatureScaler class and fitting function defined")

✓ MixedFeatureScaler class and fitting function defined


In [33]:
# --- 4.3 Apply Scaler to Single Sample ---

def apply_scaler(X: np.ndarray, scaler) -> np.ndarray:
    """
    Apply a fitted scaler (MixedFeatureScaler or StandardScaler) to a single time-series sample.
    
    Args:
        X: Time-series array of shape (T, C)
        scaler: Fitted MixedFeatureScaler or StandardScaler object
        
    Returns:
        Normalized array of shape (T, C)
    """
    if isinstance(scaler, MixedFeatureScaler):
        # MixedFeatureScaler handles the transformation directly
        return scaler.transform(X)
    else:
        # Fallback for StandardScaler (for backward compatibility)
        T, C = X.shape
        X_2d = X.reshape(-1, C)
        X_scaled = scaler.transform(X_2d)
        return X_scaled.reshape(T, C)

print("✓ Scaler application function defined")

✓ Scaler application function defined


## 5. PyTorch Dataset Class

This section defines the `HandwritingDataset` class that wraps our data for PyTorch training.

In [34]:
# --- 5.1 HandwritingDataset Class ---

class HandwritingDataset(Dataset):
    """
    PyTorch Dataset for handwriting time-series data.
    
    This class handles:
    - Loading raw time-series files
    - Computing derivative features (if enabled)
    - Normalization using a fitted scaler
    - Padding/truncation to fixed length
    - Converting to PyTorch tensors
    """
    
    def __init__(self, meta_df: pd.DataFrame, data_root: str, scaler,
                 max_len: int, use_derivatives: bool = True):
        """
        Initialize the dataset.
        
        Args:
            meta_df: DataFrame with columns ['file_name', 'subject_id', 'label']
            data_root: Root directory for data files
            scaler: Fitted MixedFeatureScaler (fit on training data only!)
            max_len: Maximum sequence length for padding/truncation
            use_derivatives: Whether to compute velocity, acceleration, jerk
        """
        self.meta_df = meta_df.reset_index(drop=True)
        self.data_root = data_root
        self.max_len = max_len
        self.use_derivatives = use_derivatives
        self.scaler = scaler
    
    def __len__(self):
        return len(self.meta_df)
    
    def __getitem__(self, idx):
        """Get a single sample from the dataset."""
        row = self.meta_df.iloc[idx]
        file_name = row['file_name']
        label = int(row['label'])
        subject_id = row['subject_id']
        
        # Try different file path patterns
        candidates = [
            os.path.join(self.data_root, f"{file_name}.svc"),
            os.path.join(self.data_root, f"{file_name}.txt"),
            os.path.join(self.data_root, str(file_name)),
        ]
        
        filepath = None
        for cp in candidates:
            if os.path.exists(cp):
                filepath = cp
                break
        
        if filepath is None:
            raise FileNotFoundError(f"Could not find raw file for sample '{file_name}'")
        
        # Load and preprocess
        ts = load_raw_timeseries(filepath)  # (T, C)
        
        if self.use_derivatives:
            ts = compute_derivatives(ts)  # (T, C+6)
        
        ts = apply_scaler(ts, self.scaler)  # Normalize
        ts_padded, length = pad_truncate(ts, self.max_len)  # (max_len, C)
        
        # Convert to (C, T) format for Conv1d (channels first)
        ts_padded = ts_padded.T  # (C, T)
        
        return {
            "x": torch.tensor(ts_padded, dtype=torch.float32),
            "y": torch.tensor(label, dtype=torch.long),
            "length": torch.tensor(length, dtype=torch.long),
            "subject_id": subject_id,
        }

print("✓ HandwritingDataset class defined")

✓ HandwritingDataset class defined


## 6. Data Splitting

This section contains functions for **subject-independent** train/test splits.

**Why subject-independent?** 
- Prevents data leakage (same subject in both train and test)
- More realistic evaluation of model generalization
- Critical for medical/clinical applications

In [35]:
# --- 6.1 Subject-Independent 3-Way Split ---

def subject_independent_split(meta_df: pd.DataFrame, train_ratio: float = 0.7,
                              val_ratio: float = 0.15, random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split data by subject IDs into Train, Validation, and Test sets.
    
    This ensures no subject appears in more than one set (subject-independent split).
    This is critical for avoiding data leakage and getting realistic performance estimates.
    
    IMPORTANT: Uses a FIXED seed (random_state) to ensure identical splits across
    all repeated runs. This allows testing model initialization stability.
    
    Args:
        meta_df: DataFrame with 'subject_id' column
        train_ratio: Proportion of subjects to use for training (default: 0.7 = 70%)
        val_ratio: Proportion of subjects to use for validation (default: 0.15 = 15%)
        random_state: Random seed for reproducibility (FIXED across all runs)
        
    Returns:
        train_df: Training set DataFrame (70% of subjects)
        val_df: Validation set DataFrame (15% of subjects)
        test_df: Test set DataFrame (15% of subjects)
    """
    subjects = meta_df['subject_id'].unique()
    rng = np.random.default_rng(random_state)
    rng.shuffle(subjects)
    
    n_train = int(len(subjects) * train_ratio)
    n_val = int(len(subjects) * val_ratio)
    
    train_subjects = set(subjects[:n_train])
    val_subjects = set(subjects[n_train:n_train + n_val])
    test_subjects = set(subjects[n_train + n_val:])
    
    train_df = meta_df[meta_df['subject_id'].isin(train_subjects)].copy()
    val_df = meta_df[meta_df['subject_id'].isin(val_subjects)].copy()
    test_df = meta_df[meta_df['subject_id'].isin(test_subjects)].copy()
    
    return train_df, val_df, test_df

print("✓ Subject-independent split function defined")

✓ Subject-independent split function defined


## 7. Prepare and Load Data

This section actually loads and prepares the data using all the functions defined above.

**Run this section after setting up your paths in the Config class!**

### Important: Check Data Directory First

Before running the data loading, make sure:
1. The data ZIP file has been **unzipped** to the `Config.DATA_ROOT` directory
2. The `Config.DATA_ROOT` path is correct
3. Files are in `.svc` or `.txt` format

In [36]:
# --- 7.0 Check Data Directory (Diagnostic) ---

print("=" * 60)
print("CHECKING DATA DIRECTORY")
print("=" * 60)
print(f"Data Root: {Config.DATA_ROOT}")
print(f"Directory exists: {os.path.exists(Config.DATA_ROOT)}")

if not os.path.exists(Config.DATA_ROOT):
    print("\n⚠ ERROR: Data directory does not exist!")
    print(f"\nPlease check:")
    print(f"1. Is the ZIP file unzipped?")
    print(f"2. Is the path in Config.DATA_ROOT correct?")
    print(f"3. Current path: {Config.DATA_ROOT}")
    print(f"\nTo unzip the data:")
    print(f"  1. Locate: dataSciRep_public.zip")
    print(f"  2. Extract it to: {os.path.dirname(Config.DATA_ROOT)}")
    print(f"  3. Or update Config.DATA_ROOT to point to the extracted folder")
else:
    # Count files
    file_count = 0
    svc_count = 0
    txt_count = 0
    for root, dirs, files in os.walk(Config.DATA_ROOT):
        for f in files:
            file_count += 1
            if f.lower().endswith('.svc'):
                svc_count += 1
            elif f.lower().endswith('.txt'):
                txt_count += 1
    
    print(f"\nFiles found in directory:")
    print(f"  Total files: {file_count}")
    print(f"  .svc files: {svc_count}")
    print(f"  .txt files: {txt_count}")
    
    if svc_count == 0 and txt_count == 0:
        print("\n⚠ WARNING: No .svc or .txt files found!")
        print("The data may not be unzipped correctly.")
    else:
        print("✓ Data directory looks good!")

print("=" * 60)

# --- 7.1 Load Metadata and Discover Files ---

# Load subject-level metadata
print("\n" + "=" * 60)
print("LOADING METADATA")
print("=" * 60)
raw_meta = pd.read_excel(Config.META_XLSX)

subject_meta = pd.DataFrame()
subject_meta['subject_id'] = raw_meta['ID'].astype(int)
subject_meta['label'] = (raw_meta['diag'].astype(str).str.upper() == 'DYSGR').astype(int)

print(f"Total subjects: {len(subject_meta)}")
print(f"Dysgraphic: {subject_meta['label'].sum()}")
print(f"Non-dysgraphic: {(subject_meta['label'] == 0).sum()}")
print("\nFirst 5 rows:")
print(subject_meta.head())

# Discover and map files to subjects
print("\n" + "=" * 60)
print("DISCOVERING FILES...")
print("=" * 60)
meta_df = discover_files_and_map_subjects(Config.DATA_ROOT, subject_meta, verbose=True)

print(f"\nTotal samples: {len(meta_df)}")
print(f"Unique subjects: {meta_df['subject_id'].nunique()}")
print(f"Dysgraphic samples: {meta_df['label'].sum()}")
print(f"Non-dysgraphic samples: {(meta_df['label'] == 0).sum()}")
print("\nFirst 5 file mappings:")
print(meta_df.head())

print("\n✓ Data preparation complete!")

CHECKING DATA DIRECTORY
Data Root: C:\Users\tiama\OneDrive\Desktop\period 2 courses\Reserach Project\DysXAI\dataSciRep_public
Directory exists: True

Files found in directory:
  Total files: 124
  .svc files: 121
  .txt files: 0
✓ Data directory looks good!

LOADING METADATA


Total subjects: 120
Dysgraphic: 57
Non-dysgraphic: 63

First 5 rows:
   subject_id  label
0           6      1
1           7      1
2           8      1
3          11      1
4          13      1

DISCOVERING FILES...
Looking for files in: C:\Users\tiama\OneDrive\Desktop\period 2 courses\Reserach Project\DysXAI\dataSciRep_public
Looking for 120 subject IDs: [6, 7, 8, 11, 13, 14, 15, 16, 17, 19]...

File Discovery Summary:
  Total .svc/.txt files found: 121
  Files with numbers in name: 121
  Files matched to subject IDs: 121

Validating files (checking channels)...
  Valid files: 121

Total samples: 121
Unique subjects: 120
Dysgraphic samples: 58
Non-dysgraphic samples: 63

First 5 file mappings:
   subject_id                                          file_name  label
0           6  dataSciRep_public\user00006\session00001\u0000...      1
1           7  dataSciRep_public\user00007\session00001\u0000...      1
2           8  dataSciRep_public\user00008\session00001\u0000...      1
3      

---

## ✅ Initialization Complete!

All functions, classes, and data are now ready. You can proceed to:

1. **`01_model_cnn1d.ipynb`** - Train the 1D CNN model
2. **`02_model_tcn.ipynb`** - Train the TCN model
3. **`03_model_cnnlstm.ipynb`** - Train the CNN-LSTM model
4. **`04_explainability.ipynb`** - Analyze feature importance and model explanations

**Important Notes:**
- The `Config` class and all functions defined here will be available in other notebooks
- Make sure to run this notebook first before training any models
- The `meta_df` variable contains all your data mappings and will be used in model training notebooks